In [1]:
!pip install -U huggingface_hub

In [7]:
from datasets import load_dataset
datasets = load_dataset("Salesforce/xlam-function-calling-60k")


xlam_function_calling_60k.json: reconstructing file:   0%|          |  0.00B / 96.1MB            

xlam_function_calling_60k.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

In [10]:
import pandas as pd

df = datasets['train'].to_pandas()

display(df.head())

,id,query,answers,tools
0,0,Where can I find live giveaways for beta acces...,"[{""name"": ""live_giveaways_by_type"", ""arguments...","[{""name"": ""live_giveaways_by_type"", ""descripti..."
1,1,I need to understand the details of the Ethere...,"[{""name"": ""web_chain_details"", ""arguments"": {""...","[{""name"": ""peers"", ""description"": ""Retrieves a..."
2,2,What is the T3MA for 'ETH/BTC' using a 1h inte...,"[{""name"": ""t3ma"", ""arguments"": {""symbol"": ""ETH...","[{""name"": ""t3ma"", ""description"": ""Fetches the ..."
3,3,List titles originally aired on networks '1' a...,"[{""name"": ""list_titles"", ""arguments"": {""networ...","[{""name"": ""get_animes"", ""description"": ""Retrie..."
4,4,Fetch the competitor standings for the recentl...,"[{""name"": ""stagecompetitorstandings"", ""argumen...","[{""name"": ""stagecompetitorstandings"", ""descrip..."


In [19]:
train_DS1 = datasets['train'].select(range(0, 5000))
test_DS1 = datasets['train'].select(range(5000, 5250))
valid_DS1 = datasets['train'].select(range(5250, 5500))

print(len(train_DS1))
print(len(test_DS1))
print(len(valid_DS1))

5000
250
250


In [20]:
!pip install -q "unsloth[colab-new]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11

In [21]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.12: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [22]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151654)
    (layers): ModuleList(
      (0-1): 2 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2R

In [23]:
def format_sample(example):
    return {
        "text": f"""Tools:
{example['tools']}

Query:
{example['query']}

Answer:
{example['answers']}"""
    }

train_formatted = train_DS1.map(format_sample)
valid_formatted = valid_DS1.map(format_sample)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

In [24]:
print(train_formatted[0]["text"])

Tools:
[{"name": "live_giveaways_by_type", "description": "Retrieve live giveaways from the GamerPower API based on the specified type.", "parameters": {"type": {"description": "The type of giveaways to retrieve (e.g., game, loot, beta).", "type": "str", "default": "game"}}}]

Query:
Where can I find live giveaways for beta access and games?

Answer:
[{"name": "live_giveaways_by_type", "arguments": {"type": "beta"}}, {"name": "live_giveaways_by_type", "arguments": {"type": "game"}}]


In [25]:
from unsloth import FastLanguageModel

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.9.12 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [26]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=valid_formatted,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    args=SFTConfig(
        output_dir="qwen-function-calling",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,
        save_total_limit=2,
        fp16=True,
        optim="adamw_8bit",
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/250 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [27]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 625
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.663885,0.712144
200,0.507226,0.622493
300,0.476101,0.574778
400,0.521829,0.535047
500,0.458770,0.504732
600,0.509051,0.486160
625,0.456114,0.484849


Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling/checkpoint-625/tokenizer_config.json.


TrainOutput(global_step=625, training_loss=0.586080816268921, metrics={'train_runtime': 3261.0525, 'train_samples_per_second': 1.533, 'train_steps_per_second': 0.192, 'total_flos': 3.3392318162190336e+16, 'train_loss': 0.586080816268921, 'epoch': 1.0})

In [29]:
sample = test_DS1[0]

prompt = f"""Tools:
{sample['tools']}

Query:
{sample['query']}

Answer:"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
)

generated = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print("QUERY:")
print(sample["query"])

print("\nEXPECTED:")
print(sample["answers"])

print("\nGENERATED:")
print(response)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY:
List all Instagram highlights for user 'user987654321'.

EXPECTED:
[{"name": "user_highlights_list", "arguments": {"userid": "user987654321"}}]

GENERATED:
 [{"name": "user_highlights_list", "arguments": {"userid": "user987654321"}}]


In [30]:
import json
import re

def normalize_json(text):
    text = text.strip()

    match = re.search(r'\[.*\]', text, re.DOTALL)

    if not match:
        return None

    try:
        return json.loads(match.group())
    except:
        return None


correct = 0
failed = 0

for sample in test_DS1:

    prompt = f"""Tools:
{sample['tools']}

Query:
{sample['query']}

Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    predicted = normalize_json(response)
    expected = normalize_json(sample["answers"])

    if predicted == expected:
        correct += 1
    else:
        failed += 1

accuracy = correct / len(test_DS1)

print(f"Correct: {correct}")
print(f"Failed: {failed}")
print(f"Exact Match Accuracy: {accuracy:.2%}")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Correct: 194
Failed: 56
Exact Match Accuracy: 77.60%


In [31]:
import json
import re

def normalize_json(text):
    text = text.strip()
    match = re.search(r'\[.*\]', text, re.DOTALL)

    if not match:
        return None

    try:
        return json.loads(match.group())
    except:
        return None


errors = []

for sample in test_DS1:

    prompt = f"""Tools:
{sample['tools']}

Query:
{sample['query']}

Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    predicted = normalize_json(response)
    expected = normalize_json(sample["answers"])

    if predicted != expected:
        errors.append({
            "query": sample["query"],
            "expected": expected,
            "predicted": predicted
        })

print("Number of errors:", len(errors))

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Number of errors: 56


In [32]:
for i, error in enumerate(errors[:5]):
    print("=" * 80)
    print("QUERY:", error["query"])
    print("\nEXPECTED:", error["expected"])
    print("\nPREDICTED:", error["predicted"])

QUERY: Retrieve information for product '75319' in Australian Dollars and Italian language.

EXPECTED: [{'name': 'products_detail', 'arguments': {'goods_id': '75319', 'currency': 'AUD', 'language': 'it'}}]

PREDICTED: [{'name': 'products_detail', 'arguments': {'goods_id': '75319', 'currency': 'AUD', 'country': 'AU', 'language': 'it'}}]
QUERY: What are the chances of rolling exactly 20 with five 6-sided dice?

EXPECTED: [{'name': 'dice_roll_probability', 'arguments': {'target_sum': 20, 'num_dice': 5, 'num_faces': 6}}]

PREDICTED: [{'name': 'dice_roll_probability', 'arguments': {'target_sum': 20, 'num_dice': 5}}]
QUERY: Simulate querying the 'sales' table with conditions: item='laptop', price>1000.

EXPECTED: [{'name': 'simulate_query_database', 'arguments': {'table': 'sales', 'conditions': [{'item': 'laptop', 'price': {'$gt': 1000}}]}}]

PREDICTED: [{'name': 'simulate_query_database', 'arguments': {'table': 'sales', 'conditions': [{'item': 'laptop', 'price': '>1000'}]}}]
QUERY: What are

In [39]:
train_DS2 = datasets["train"].select(range(10000, 15000))
valid_DS2 = datasets["train"].select(range(15000, 15250))
test_DS2  = datasets["train"].select(range(15250, 15500))

print(len(train_DS2))
print(len(valid_DS2))
print(len(test_DS2))

5000
250
250


In [40]:
train_formatted2 = train_DS2.map(format_sample)
valid_formatted2 = valid_DS2.map(format_sample)
test_formatted2 = test_DS2.map(format_sample)

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

In [41]:
trainer2 = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted2,
    eval_dataset=valid_formatted2,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    args=SFTConfig(
        output_dir="qwen-function-calling-10k",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        fp16=True,
        optim="adamw_8bit",
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/250 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [42]:
trainer2.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 625
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
10,0.453022
20,0.442461
30,0.537747
40,0.491885
50,0.473395
60,0.460812
70,0.462830
80,0.432773
90,0.447047
100,0.431041


Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling-10k/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling-10k/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling-10k/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling-10k/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling-10k/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling-10k/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in qwen-function-calling-10k/checkpoint-625/tokenizer_config.json.


TrainOutput(global_step=625, training_loss=0.4054420873641968, metrics={'train_runtime': 2913.5699, 'train_samples_per_second': 1.716, 'train_steps_per_second': 0.215, 'total_flos': 3.300683357761536e+16, 'train_loss': 0.4054420873641968, 'epoch': 1.0})

In [44]:
import json
import re

# ============================================================
# TEST SET
# ============================================================

test_DS2 = datasets["train"].select(range(15000, 15250))


# ============================================================
# JSON PARSER
# ============================================================

def extract_json(text):
    text = text.strip()

    # Find the first JSON array
    start = text.find("[")
    end = text.rfind("]")

    if start == -1 or end == -1:
        return None

    json_text = text[start:end + 1]

    try:
        result = json.loads(json_text)

        # Make sure result is a list
        if isinstance(result, list):
            return result

        return None

    except:
        return None


# ============================================================
# SAFE TOOL CALL NORMALIZATION
# ============================================================

def normalize_calls(data):

    if data is None:
        return []

    # Sometimes nested list
    while (
        isinstance(data, list)
        and len(data) == 1
        and isinstance(data[0], list)
    ):
        data = data[0]

    if not isinstance(data, list):
        return []

    calls = []

    for item in data:

        # Skip malformed items
        if not isinstance(item, dict):
            continue

        name = item.get("name")

        arguments = item.get("arguments", {})

        if not isinstance(arguments, dict):
            arguments = {}

        calls.append({
            "name": name,
            "arguments": arguments
        })

    return calls


# ============================================================
# EVALUATION
# ============================================================

correct = 0
failed = 0

tool_correct = 0
tool_total = 0

argument_correct = 0
argument_total = 0

errors = []


for i, sample in enumerate(test_DS2):

    prompt = f"""Tools:
{sample['tools']}

Query:
{sample['query']}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    predicted_raw = extract_json(response)
    expected_raw = extract_json(sample["answers"])

    predicted = normalize_calls(predicted_raw)
    expected = normalize_calls(expected_raw)


    # ========================================================
    # EXACT MATCH
    # ========================================================

    if predicted == expected:

        correct += 1

    else:

        failed += 1

        errors.append({
            "index": i,
            "query": sample["query"],
            "expected": expected,
            "predicted": predicted,
            "raw_output": response
        })


    # ========================================================
    # TOOL + ARGUMENT METRICS
    # ========================================================

    max_len = max(len(expected), len(predicted))

    for j in range(max_len):

        # Missing predicted call
        if j >= len(predicted):
            continue

        # Extra predicted call
        if j >= len(expected):
            continue

        expected_call = expected[j]
        predicted_call = predicted[j]

        # -----------------------------
        # Tool name
        # -----------------------------

        tool_total += 1

        if (
            expected_call["name"]
            == predicted_call["name"]
        ):
            tool_correct += 1


        # -----------------------------
        # Arguments
        # -----------------------------

        argument_total += 1

        if (
            expected_call["arguments"]
            == predicted_call["arguments"]
        ):
            argument_correct += 1


# ============================================================
# METRICS
# ============================================================

total_samples = len(test_DS2)

exact_accuracy = (
    correct / total_samples * 100
)

tool_accuracy = (
    tool_correct / tool_total * 100
    if tool_total > 0 else 0
)

argument_accuracy = (
    argument_correct / argument_total * 100
    if argument_total > 0 else 0
)


# ============================================================
# SUMMARY
# ============================================================

print("=" * 80)
print("                    EVALUATION RESULTS")
print("=" * 80)

print(f"Total Test Samples      : {total_samples}")
print(f"Correct                 : {correct}")
print(f"Failed                  : {failed}")

print("-" * 80)

print(f"Exact Match Accuracy    : {exact_accuracy:.2f}%")

print("-" * 80)

print(f"Tool Calls Evaluated    : {tool_total}")
print(f"Correct Tool Names      : {tool_correct}")
print(f"Tool Name Accuracy      : {tool_accuracy:.2f}%")

print("-" * 80)

print(f"Arguments Evaluated     : {argument_total}")
print(f"Correct Arguments       : {argument_correct}")
print(f"Argument Accuracy       : {argument_accuracy:.2f}%")

print("-" * 80)

print(f"Total Errors            : {len(errors)}")

print("=" * 80)


# ============================================================
# DETAILED ERRORS
# ============================================================

print("\n")
print("=" * 80)
print("                         ERRORS")
print("=" * 80)


for n, error in enumerate(errors, 1):

    print("\n" + "=" * 80)

    print(
        f"ERROR {n}  |  TEST INDEX: {error['index']}"
    )

    print("=" * 80)

    print("\nQUERY:")
    print(error["query"])


    print("\nEXPECTED:")

    print(
        json.dumps(
            error["expected"],
            ensure_ascii=False,
            indent=2
        )
    )


    print("\nPREDICTED:")

    print(
        json.dumps(
            error["predicted"],
            ensure_ascii=False,
            indent=2
        )
    )


    # ========================================================
    # DIFFERENCE ANALYSIS
    # ========================================================

    print("\nDIFFERENCES:")

    expected_calls = error["expected"]
    predicted_calls = error["predicted"]

    max_calls = max(
        len(expected_calls),
        len(predicted_calls)
    )


    for j in range(max_calls):

        # -----------------------------------------
        # Missing call
        # -----------------------------------------

        if j >= len(predicted_calls):

            print(
                f"\n  Call {j + 1}: MISSING TOOL CALL"
            )

            print(
                "    Expected:",
                expected_calls[j]
            )

            continue


        # -----------------------------------------
        # Extra call
        # -----------------------------------------

        if j >= len(expected_calls):

            print(
                f"\n  Call {j + 1}: EXTRA TOOL CALL"
            )

            print(
                "    Predicted:",
                predicted_calls[j]
            )

            continue


        expected_call = expected_calls[j]
        predicted_call = predicted_calls[j]


        # -----------------------------------------
        # Tool name difference
        # -----------------------------------------

        if (
            expected_call["name"]
            != predicted_call["name"]
        ):

            print(
                f"\n  Call {j + 1}: TOOL NAME DIFFERENCE"
            )

            print(
                f"    Expected : {expected_call['name']}"
            )

            print(
                f"    Predicted: {predicted_call['name']}"
            )


        # -----------------------------------------
        # Arguments
        # -----------------------------------------

        expected_args = expected_call["arguments"]
        predicted_args = predicted_call["arguments"]


        missing_args = (
            set(expected_args)
            - set(predicted_args)
        )

        extra_args = (
            set(predicted_args)
            - set(expected_args)
        )


        if missing_args:

            print(
                f"\n  Call {j + 1}: MISSING ARGUMENTS"
            )

            print(
                f"    {missing_args}"
            )


        if extra_args:

            print(
                f"\n  Call {j + 1}: EXTRA ARGUMENTS"
            )

            print(
                f"    {extra_args}"
            )


        # -----------------------------------------
        # Wrong argument values
        # -----------------------------------------

        common_args = (
            set(expected_args)
            & set(predicted_args)
        )


        for key in common_args:

            if (
                expected_args[key]
                != predicted_args[key]
            ):

                print(
                    f"\n  Call {j + 1}: WRONG ARGUMENT VALUE"
                )

                print(
                    f"    Argument : {key}"
                )

                print(
                    f"    Expected : {expected_args[key]}"
                )

                print(
                    f"    Predicted: {predicted_args[key]}"
                )

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

                    EVALUATION RESULTS
Total Test Samples      : 250
Correct                 : 180
Failed                  : 70
--------------------------------------------------------------------------------
Exact Match Accuracy    : 72.00%
--------------------------------------------------------------------------------
Tool Calls Evaluated    : 365
Correct Tool Names      : 363
Tool Name Accuracy      : 99.45%
--------------------------------------------------------------------------------
Arguments Evaluated     : 365
Correct Arguments       : 294
Argument Accuracy       : 80.55%
--------------------------------------------------------------------------------
Total Errors            : 70


                         ERRORS

ERROR 1  |  TEST INDEX: 13

QUERY:
Please provide details for the fitness video 'Morning Yoga for Beginners' and the gaming video 'Fortnite World Cup Highlights'?

EXPECTED:
[
  {
    "name": "video",
    "arguments": {
      "is_id": "8J12b16Hwq0"
    }
  },
  {
 

In [46]:
model.push_to_hub_merged(
    "FatalHub/qwen-function-calling-10k",
    tokenizer,
    save_method="merged_16bit",
)

Unsloth: Restored added_tokens_decoder metadata in FatalHub/qwen-function-calling-10k/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  601kB / 2.20GB            

Validating                    :           |   0%            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:15<00:00, 97.94s/it]


Unsloth: Merge process complete. Saved to `/content/FatalHub/qwen-function-calling-10k`
